In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Input, Conv2D, LeakyReLU, Add, AveragePooling2D, MaxPooling2D
from tensorflow.keras.models import Model

def build_and_run_model():
    input_data = np.array([
        [1, 2, 3, 0, 1],
        [0, 1, 2, 3, 0],
        [3, 0, 1, 2, 1],
        [1, 2, 3, 0, 0],
        [0, 1, 0, 1, 2]
    ], dtype=np.float32).reshape(1, 5, 5, 1)

    print("Input Data Shape:", input_data.shape)
    print("Input Data Matrix:\n", input_data[0, :, :, 0])
    print("-" * 50)

    input_tensor = Input(shape=(5, 5, 1))

    x = Conv2D(filters=1, kernel_size=(3, 3), padding='same',
               kernel_initializer='ones', use_bias=False, name="Initial_Conv")(input_tensor)
    x = LeakyReLU(name="Initial_LeakyReLU")(x)

    path_a = x

    path_b = Conv2D(filters=1, kernel_size=(3, 3), dilation_rate=3, padding='same',
                    kernel_initializer='ones', use_bias=False, name="Dilated_Conv_r3")(x)
    path_b = LeakyReLU(name="LeakyReLU_r3")(path_b)

    path_b = Conv2D(filters=1, kernel_size=(3, 3), dilation_rate=2, padding='same',
                    kernel_initializer='ones', use_bias=False, name="Dilated_Conv_r2")(path_b)
    path_b = LeakyReLU(name="LeakyReLU_r2")(path_b)

    path_b = Conv2D(filters=1, kernel_size=(3, 3), dilation_rate=1, padding='same',
                    kernel_initializer='ones', use_bias=False, name="Dilated_Conv_r1")(path_b)
    path_b = LeakyReLU(name="LeakyReLU_r1")(path_b)

    merged = Add(name="Merge_Add")([path_a, path_b])

    intermediate = Conv2D(filters=1, kernel_size=(3, 3), padding='same',
                          kernel_initializer='ones', use_bias=False, name="Intermediate_Conv")(merged)

    pool_avg = AveragePooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid', name="Avg_Pool")(intermediate)

    pool_max = MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid', name="Max_Pool")(intermediate)

    final_output = Add(name="Final_Output")([pool_avg, pool_max])

    model = Model(inputs=input_tensor, outputs=final_output)

    print("\nModel Summary:")
    model.summary()

    prediction = model.predict(input_data)

    print("\n" + "-" * 50)
    print("FINAL OUTPUT FEATURE MAP VALUES")
    print("-" * 50)
    print(prediction[0, :, :, 0])

if __name__ == "__main__":
    build_and_run_model()


Input Data Shape: (1, 5, 5, 1)
Input Data Matrix:
 [[1. 2. 3. 0. 1.]
 [0. 1. 2. 3. 0.]
 [3. 0. 1. 2. 1.]
 [1. 2. 3. 0. 0.]
 [0. 1. 0. 1. 2.]]
--------------------------------------------------

Model Summary:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 5, 5, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Initial_Conv        │ (None, 5, 5, 1)   │          9 │ input_layer_1[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Initial_LeakyReLU   │ (None, 5, 5, 1)   │          0 │ Initial_Conv[0][… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dilated_Conv_r3     │ (None, 5, 5, 1)   │          9 │ Initial_LeakyReL… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LeakyReLU_r3        │ (None, 5, 5, 1)   │          0 │ Dilated_Conv_r3[… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dilated_Conv_r2     │ (None, 5, 5, 1)   │          9 │ LeakyReLU_r3[0][… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LeakyReLU_r2        │ (None, 5, 5, 1)   │          0 │ Dilated_Conv_r2[… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dilated_Conv_r1     │ (None, 5, 5, 1)   │          9 │ LeakyReLU_r2[0][… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LeakyReLU_r1        │ (None, 5, 5, 1)   │          0 │ Dilated_Conv_r1[… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Merge_Add (Add)     │ (None, 5, 5, 1)   │          0 │ Initial_LeakyReL… │
│                     │                   │            │ LeakyReLU_r1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Intermediate_Conv   │ (None, 5, 5, 1)   │          9 │ Merge_Add[0][0]   │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Avg_Pool            │ (None, 2, 2, 1)   │          0 │ Intermediate_Con… │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Max_Pool            │ (None, 2, 2, 1)   │          0 │ Intermediate_Con… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Final_Output (Add)  │ (None, 2, 2, 1)   │          0 │ Avg_Pool[0][0],   │
│                     │                   │            │ Max_Pool[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 45 (180.00 B)

 Trainable params: 45 (180.00 B)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step

--------------------------------------------------
FINAL OUTPUT FEATURE MAP VALUES
--------------------------------------------------
[[13836.5  17082.75]
 [17054.25 21165.  ]]
